# Policy Gradient — REINFORCE from Scratch Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: softmax policy network

In [ ]:
```python

def policy_logits(theta, state_features):

    return [dot(theta[a], state_features) for a in range(N_ACTIONS)]

def softmax(logits):

    m = max(logits)

    exps = [exp(l - m) for l in logits]

    Z = sum(exps)

    return [e / Z for e in exps]

In [ ]:
```

Use a linear policy (one weight vector per action) for a tabular env. For Atari, swap in a CNN and keep the softmax head.

### Step 2: sampling and log-probability

In [ ]:
```python

def sample_action(probs, rng):

    x = rng.random()

    cum = 0

    for a, p in enumerate(probs):

        cum += p

        if x <= cum:

            return a

    return len(probs) - 1

def log_prob(probs, a):

    return log(probs[a] + 1e-12)

In [ ]:
```

### Step 3: rollout with log-probs captured

In [ ]:
```python

def rollout(theta, env, rng, gamma):

    trajectory = []

    s = env.reset()

    while not done:

        logits = policy_logits(theta, s)

        probs = softmax(logits)

        a = sample_action(probs, rng)

        s_next, r, done = env.step(s, a)

        trajectory.append((s, a, r, probs))

        s = s_next

    return trajectory

In [ ]:
```

### Step 4: REINFORCE update

In [ ]:
```python

def reinforce_step(theta, trajectory, gamma, lr, baseline=0.0):

    returns = compute_returns(trajectory, gamma)

    for (s, a, _, probs), G in zip(trajectory, returns):

        advantage = G - baseline

        grad_log_pi_a = [-p for p in probs]

        grad_log_pi_a[a] += 1.0

        for i in range(N_ACTIONS):

            for j in range(len(s)):

                theta[i][j] += lr * advantage * grad_log_pi_a[i] * s[j]

In [ ]:
```

The gradient `∇ log π(a|s) = e_a - π(·|s)` (onehot of `a` minus probabilities) is the heart of softmax policy gradients. Burn it into muscle memory.

### Step 5: baselines

A running mean of `G` over recent episodes is enough variance reduction to get a 4×4 GridWorld running; it takes ~500 episodes to converge. Upgrade the baseline to a learned `V̂(s)` and you get actor-critic.

## Exercises

In [ ]:
1. **Easy.** Implement REINFORCE on 4×4 GridWorld with a linear softmax policy. Train for 1,000 episodes without a baseline. Plot the learning curve; measure variance (std of returns).
2. **Medium.** Add a running-mean baseline. Train again. Compare sample efficiency and variance to the vanilla run. By how much does the baseline reduce steps to convergence?
3. **Hard.** Add an entropy bonus `β · H(π)`. Sweep `β ∈ {0, 0.01, 0.1, 1.0}`. Plot final return and policy entropy. Where is the sweet spot on this task?